In [ ]:
import gzip
import csv
import time

ARQUIVO_ENTRADA = 'wdump-5306.nt.gz'
ARQUIVO_SAIDA = 'wikidata_completo_bruto.csv'

COLUNAS = [
    'id', 'nome', 
    'genero_P21', 'nacionalidade_P27', 'nome_nascimento_P1477', 'sobrenome_P734', 'nome_P735',
    'nascimento_data_P569', 'morte_data_P570', 'local_nascimento_P19',
    'pai_P22', 'mae_P25', 'irmaos_P3373', 'conjugue_P26', 'filhos_P40',
    'parente_P1038', 'ocupacao_P106', 'residencia_P551', 'email_P968'
]

MAPA_P = {
    'P21': 'genero_P21', 'P27': 'nacionalidade_P27', 'P1477': 'nome_nascimento_P1477', 'P734': 'sobrenome_P734', 'P735': 'nome_P735',
    'P569': 'nascimento_data_P569', 'P570': 'morte_data_P570', 'P19': 'local_nascimento_P19',
    'P22': 'pai_P22', 'P25': 'mae_P25', 'P3373': 'irmaos_P3373',
    'P26': 'conjugue_P26', 'P40': 'filhos_P40', 'P1038': 'parente_P1038',
    'P106': 'ocupacao_P106', 'P551': 'residencia_P551', 'P968': 'email_P968'
}

PREDICADOS_DE_NOME = {
    '<http://www.w3.org/2000/01/rdf-schema#label>',
    '<http://schema.org/name>',
    '<http://www.w3.org/2004/02/skos/core#prefLabel>',
}

def limpar_string(texto_bruto):
    """Remove aspas e extrai a lingua."""
    if not texto_bruto: return "", ""
    if texto_bruto.startswith('"'):
        idx_fim = texto_bruto.rfind('"')
        if idx_fim > 0:
            texto = texto_bruto[1:idx_fim]
            meta = texto_bruto[idx_fim+1:].replace('@', '')
            return texto, meta
    return texto_bruto, ""

def escolher_melhor_nome(lista_nomes):
    """Prioridade: Inglês > Dialetos EN > Sem Lingua > Outros"""
    if not lista_nomes: return None
    
    melhor_nome = lista_nomes[0][0]
    melhor_score = -1
    
    for nome, lingua in lista_nomes:
        score = 0
        if lingua == 'en': score = 100
        elif lingua.startswith('en-'): score = 90
        elif lingua == '': score = 50
        else: score = 1
            
        if score > melhor_score:
            melhor_score = score
            melhor_nome = nome
    return melhor_nome

print(f"Iniciando Extração Completa (Sem Limites)...")
start = time.time()

with gzip.open(ARQUIVO_ENTRADA, 'rt', encoding='utf-8') as fin, \
     open(ARQUIVO_SAIDA, 'w', encoding='utf-8', newline='') as fout:

    writer = csv.DictWriter(fout, fieldnames=COLUNAS)
    writer.writeheader()

    atual_id = None
    dados = {c: None for c in COLUNAS}
    candidatos_nome = [] 
    count = 0

    for linha in fin:
        if not linha or linha.startswith('#'): continue

        partes = linha.split(' ', 2)
        if len(partes) < 3: continue
        
        sujeito_raw, pred_raw, obj_raw = partes
        obj_raw = obj_raw.rsplit(' .', 1)[0] 

        if '/entity/Q' not in sujeito_raw: continue
        this_id = sujeito_raw.split('/entity/')[1].replace('>', '')

        if this_id != atual_id:
            if atual_id is not None:
                # tenta achar o melhor nome nos labels
                nome_final = escolher_melhor_nome(candidatos_nome)
                
                # se não achou nome, mas tem Nome de Nascimento (P1477), usa ele
                if not nome_final and dados['nome_nascimento_P1477']:
                    # pega o primeiro nome da lista de nascimento
                    nome_final = dados['nome_nascimento_P1477'][0]
                
                dados['nome'] = nome_final

                row = {k: (';'.join(v) if isinstance(v, list) else v) for k, v in dados.items()}
                writer.writerow(row)
                count += 1
                if count % 100000 == 0:
                    print(f"Processados: {count/1000000:.1f} Milhões...", end='\r')

            # reseta variáveis
            atual_id = this_id
            dados = {c: None for c in COLUNAS}
            dados['id'] = atual_id
            candidatos_nome = [] 
        
        if pred_raw in PREDICADOS_DE_NOME:
            texto, lingua = limpar_string(obj_raw)
            if texto:
                candidatos_nome.append((texto, lingua))
        
        elif '/prop/direct/P' in pred_raw:
            p_code = pred_raw.split('/prop/direct/')[1].replace('>', '')
            
            if p_code in MAPA_P:
                coluna = MAPA_P[p_code]
                
                val = obj_raw
                if '/entity/' in val:
                    val = val.split('/entity/')[1].replace('>', '')
                else:
                    val, _ = limpar_string(val)
                
                if dados[coluna] is None:
                    dados[coluna] = [val]
                elif val not in dados[coluna]:
                    dados[coluna].append(val)

    # salva o último registro do arquivo
    if atual_id:
        nome_final = escolher_melhor_nome(candidatos_nome)
        if not nome_final and dados['nome_nascimento_P1477']:
            nome_final = dados['nome_nascimento_P1477'][0]
        dados['nome'] = nome_final
        row = {k: (';'.join(v) if isinstance(v, list) else v) for k, v in dados.items()}
        writer.writerow(row)

print(f"\n\nCONCLUÍDO! Arquivo {ARQUIVO_SAIDA} gerado com sucesso.")

In [ ]:
import polars as pl

ARQUIVO_ENTRADA = "wikidata_completo_bruto.csv"

print("🚀 Carregando o monstro (CSV Gigante)...")
# low_memory=True ajuda a não estourar a RAM no carregamento inicial
df = pl.read_csv(ARQUIVO_ENTRADA, ignore_errors=True)

print(f"Total Inicial: {len(df)} pessoas.")

print("🔪 Removendo pessoas sem vínculos familiares...")

# Define as colunas de família
cols_familia = ["pai_P22", "mae_P25", "irmaos_P3373", "conjugue_P26", "filhos_P40"]

# mantém a linha se pelo menos uma dessas colunas não for nula
df_conectado = df.filter(
    pl.any_horizontal(pl.col(cols_familia).is_not_null())
)

qtd_removida = len(df) - len(df_conectado)
print(f"Removidos {qtd_removida} registros 'solitários'.")
print(f"Restaram: {len(df_conectado)} pessoas conectadas.")

df_conectado.write_csv("wikidata_limpo.csv")
print("Salvo com sucesso!")

In [ ]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt

NOME_DO_ARQUIVO = 'wikidata_limpo.csv'

df_wikidata_limpo = pd.read_csv(NOME_DO_ARQUIVO)

msno.matrix(df_wikidata_limpo, figsize=(6, 3))
plt.title(f'Valores Nulos')
plt.xticks(fontsize=8)
plt.show()

df_wikidata_limpo.head()

In [ ]:
import polars as pl
ARQUIVO_ENTRADA = "wikidata_completo_bruto.csv" # "wikidata_limpo.csv"
df = pl.read_csv(ARQUIVO_ENTRADA, ignore_errors=True)

cols_familia = ["nascimento_data_P569"]

df_conectado = df.filter(
    pl.any_horizontal(pl.col(cols_familia).is_not_null())
)

"""
cols_familia = ["residencia_P551", "ocupacao_P106"]

df_conectado = df_conectado.filter(
    pl.any_horizontal(pl.col(cols_familia).is_not_null())
)
"""

df_conectado.write_csv("wikidata8.csv") # wikidata8

In [ ]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt

NOME_DO_ARQUIVO = 'wikidata8.csv'

df = pd.read_csv(NOME_DO_ARQUIVO)

msno.matrix(df, figsize=(6, 3))
plt.title(f'Valores Nulos')
plt.xticks(fontsize=8)
plt.show()

df.head()